# Final Capstone: Retail Demand and Customer Insights

This final capstone is intentionally more detailed than the earlier notebooks. Students are asked to define the target, validate the data, evaluate a model, and explain its limitations in a business-style memo.


## Project setup

The project dataset is synthetic but realistic: retail orders, categories, discount percentages, and region-level variation. Students should not treat the dataset as magic; it must be explored and documented.


In [ ]:
import pandas as pd
import numpy as np

orders = pd.read_csv('static/notebooks/data-science/datasets/capstone_orders.csv')
print(orders.head())
print(orders.shape)
assert len(orders) == 320


## Data quality checks

A final capstone should always begin with validation. A model is only as trustworthy as the quality of the underlying dataset.


In [ ]:
assert orders['quantity'].gt(0).all()
assert orders['unit_price'].gt(0).all()
assert orders['discount_pct'].between(0, 20).all()
assert orders['revenue'].notna().all()
print('missing values', orders.isna().sum().to_dict())


## Exploratory analysis

The aim here is not to overfit to a made-up story. It is to see the patterns, understand the signal, and decide which features are meaningful before modeling.


In [ ]:
region_summary = orders.groupby('region')['revenue'].agg(['count', 'sum', 'mean', 'median'])
print(region_summary.round(2))
assert region_summary['count'].sum() == len(orders)


## Model and evaluation

The capstone must include a baseline, a learned model, and a clear explanation of the metric choice. A solid business report does not hide the decision logic.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

orders['high_value'] = (orders['revenue'] >= orders['revenue'].median()).astype(int)
X = orders[['quantity', 'unit_price', 'discount_pct', 'region', 'weekend', 'customer_rating']]
y = orders['high_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
num = ['quantity', 'unit_price', 'discount_pct', 'weekend', 'customer_rating']
cat = ['region']
pre = ColumnTransformer([('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num), ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat)])
base = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
model = make_pipeline(pre, LogisticRegression(max_iter=2000, random_state=42)).fit(X_train, y_train)
base_pred = base.predict(X_test)
model_pred = model.predict(X_test)
print('baseline accuracy', round(accuracy_score(y_test, base_pred), 3))
print('model accuracy', round(accuracy_score(y_test, model_pred), 3))
print('model F1', round(f1_score(y_test, model_pred, zero_division=0), 3))
assert len(model_pred) == len(y_test)


## Submission memo checklist

The final deliverable is a short memo with a business question, a brief data-quality section, key findings, model performance, and limitations. The submission should sound like a professional recommendation rather than a toy lab answer.


In [ ]:
print('Capstone checks passed. Summarize the business question, evidence, and limitations in the markdown below.')
assert True
